# QUESTION 2 Total Marks [18]

Notebook ordered to the **template** headings 2.1 → 2.2 → 2.3 → 2.4. Runs on the frozen Drive splits (`train.parquet`, `val.parquet`, `test.parquet`, `prep.py`).

- Figures save to `/cursor/stores/self/media/q2/` (and `figures/` if that folder exists).
- EDA uses **train only**. Test is counted in 2.3 for the split table, not plotted, not used to fit.
- Does **not** train Logistic Regression, Random Forest, MLP, or the embedding net.

Report prose: `/cursor/stores/self/docs/q2.md`.


In [ ]:
from pathlib import Path
import importlib.util
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[misc,assignment]

CLASSES = ["Resolved Fast", "Resolved Slow", "Unresolved"]
SEED = 42


def find_data_dir() -> Path:
    """Frozen Drive splits: team-artifacts first, then repo data/processed."""
    candidates = [
        Path("/cursor/stores/self/internal/team-artifacts"),
        Path("/workspace/data/processed"),
        Path("data/processed"),
        Path("../data/processed"),
    ]
    for p in candidates:
        if (p / "train.parquet").exists():
            return p.resolve()
    raise FileNotFoundError(
        "Missing train.parquet. Put Drive splits in data/processed/ "
        "or keep them in /cursor/stores/self/internal/team-artifacts/."
    )


def load_prep(data_dir: Path):
    prep_candidates = [
        data_dir / "prep.py",
        Path("/workspace/src/prep.py"),
        Path("src/prep.py"),
        Path("../src/prep.py"),
    ]
    for path in prep_candidates:
        if path.exists():
            spec = importlib.util.spec_from_file_location("c3_prep", path)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError("Could not find prep.py next to the splits or in src/.")


def figure_dirs() -> list[Path]:
    dirs = []
    store = Path("/cursor/stores/self/media/q2")
    try:
        store.mkdir(parents=True, exist_ok=True)
        dirs.append(store)
    except OSError:
        pass
    repo = Path("figures")
    if not repo.exists():
        alt = Path("../figures")
        repo = alt if alt.parent.exists() else repo
    repo.mkdir(parents=True, exist_ok=True)
    dirs.append(repo.resolve())
    # unique
    out, seen = [], set()
    for d in dirs:
        if d not in seen:
            out.append(d)
            seen.add(d)
    return out


DATA = find_data_dir()
prep = load_prep(DATA)
print("data_dir:", DATA)
print("prep module:", prep.__file__)
print("SEED", prep.SEED, "FEATURES", prep.FEATURES)
print("BANNED ∩ FEATURES", prep.BANNED & set(prep.FEATURES))


In [ ]:
train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "val.parquet")
print("train", len(train), "val", len(val))
print("columns", list(train.columns))


## 2.1 Create a data dictionary and audit class balance, duplicates, missingness, invalid values, outliers and possible leakage. (4)


### Data dictionary

Modelling table (Drive splits). Raw portal columns that were dropped before the split are listed in the audit markdown below — they are **not** in these parquets.


In [ ]:
dictionary = pd.DataFrame([
    ["C3_Complaint_Type", "text", "Fault / request type", "Feature. <100 train rows → infrequent"],
    ["department", "text", "9 groups from Work_Center", "Feature. Raw work centre banned"],
    ["suburb_group", "text", "Suburb names with ≥500 train rows; else OTHER", "Feature. Raw Suburb banned"],
    ["Ward", "text", 'Ward "1"–"116"', "Feature. Placeholders → MISSING"],
    ["Sub_Council", "text", "Sub-Council 1–24", "Feature. Same missing handling"],
    ["Created_On_Date", "datetime", "Log date (Africa/Johannesburg date)", "Feature source: month, weekday, weekend"],
    ["target", "text", "Resolved Fast / Slow / Unresolved", "Label. 7-day cut. Never a feature"],
], columns=["column", "type", "meaning", "use"])
dictionary


### Class balance, duplicates, missingness, invalid values, outliers, leakage

Duplicates, invalid dates and resolution-time outliers were removed **before** these splits (raw 5,199,661 → modelling 3,029,242; see `q2.md` row flow). The modelling table has no `Notification` id, so raw duplicate counts cannot be recomputed here. Code audits **what is in train**.


In [ ]:
n = len(train)
vc = train["target"].astype(str).value_counts().reindex(CLASSES)
balance = pd.DataFrame({
    "n": vc,
    "share_%": 100 * vc / n,
})
print("CLASS BALANCE (train)")
display(balance.round(2))

# Duplicates in the modelling table (Notification already collapsed/dropped)
print("duplicate feature-rows (excl. target):",
      int(train[prep.FEATURES].duplicated().sum()))

ward_na = train["Ward"].isna() | train["Ward"].astype(str).str.strip().isin(["", "#", "nan", "<NA>", "None"])
sc_na = train["Sub_Council"].isna() | train["Sub_Council"].astype(str).str.strip().isin(["", "#", "nan", "<NA>", "None"])
sg = train["suburb_group"].astype(str)
missingness = pd.Series({
    "Ward missing %": 100 * ward_na.mean(),
    "Sub_Council missing %": 100 * sc_na.mean(),
    "suburb_group '#' %": 100 * (sg == "#").mean(),
    "suburb_group OTHER %": 100 * (sg == "OTHER").mean(),
    "complaint type missing %": 100 * train["C3_Complaint_Type"].isna().mean(),
})
print("\nMISSINGNESS (train)")
print(missingness.round(2).to_string())

created = pd.to_datetime(train["Created_On_Date"])
print("\nINVALID / RANGE (train created dates)")
print("min", created.min(), "max", created.max())
print("null created", int(train["Created_On_Date"].isna().sum()))
print("null target", int(train["target"].isna().sum()))

print("\nCARDINALITY (train nunique, incl. NA)")
print({c: int(train[c].nunique(dropna=False)) for c in train.columns})

print("\nLEAKAGE")
print("BANNED ∩ columns:", sorted(prep.BANNED & set(train.columns)))
print("FEATURES present:", all(c in train.columns for c in prep.FEATURES))
print("BANNED ∩ FEATURES (must be empty):", prep.BANNED & set(prep.FEATURES))
print("Locked resolution-time outliers (audit, not in this table): median 2d, p75=7d, p90=34d, p95=82d, max ~1135d — kept as Slow.")


Locked raw-file audit (cannot recompute without `Notification` / `Completed_Date`): 941,943 repeated Notification rows; 933,064 removed after invalid dates; 8,899 invalid dates removed; placeholders recoded. Full row flow is in `/cursor/stores/self/docs/q2.md`.


## 2.2 Present at least five purposeful visualisations: target distribution; missingness/data quality; two feature–target relationships; and one geographic, temporal or subgroup view. Each figure must have a title, units, readable labels and an interpretation. (6)

Train only. Five files:

1. `fig01_target_distribution.png`
2. `fig02_missingness_quality.png`
3. `fig03_department_class_mix.png` (feature–target 1)
4. `fig04_complaint_type_class_mix.png` (feature–target 2)
5. `fig05_temporal_train.png` (temporal view)


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

OUTS = figure_dirs()
print("save to", OUTS)
COLORS = ["#2ca02c", "#ff7f0e", "#d62728"]
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "figure.facecolor": "white"})

def save(fig, name):
    for d in OUTS:
        fig.savefig(d / name, dpi=160)
    plt.close(fig)
    print("wrote", name)

n = len(train)
vc = train["target"].astype(str).value_counts().reindex(CLASSES)

# 2.2.1 target distribution
fig, ax = plt.subplots(figsize=(8.2, 5.2))
bars = ax.bar(CLASSES, vc.values, color=COLORS, width=0.72)
ax.set_title(f"Figure 2.2.1 Target distribution (train, n={n:,}; seed 42)")
ax.set_xlabel("Resolution class")
ax.set_ylabel("Number of service requests")
ax.set_ylim(0, vc.max() * 1.15)
for b, c in zip(bars, vc.values):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 20000,
            f"{int(c):,}\n({100 * c / n:.1f}%)", ha="center", va="bottom", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
save(fig, "fig01_target_distribution.png")


**Interpretation (2.2.1).** Fast is 67.4% of train. A most-frequent dummy will look strong on accuracy and fail Slow/Unresolved. Macro-F1 is the primary metric.


In [ ]:
# 2.2.2 missingness / data quality
ward_na = train["Ward"].isna() | train["Ward"].astype(str).str.strip().isin(["", "#", "nan", "<NA>", "None"])
sc_na = train["Sub_Council"].isna() | train["Sub_Council"].astype(str).str.strip().isin(["", "#", "nan", "<NA>", "None"])
sg = train["suburb_group"].astype(str)
miss = pd.Series({
    "Ward missing": 100 * ward_na.mean(),
    "Sub_Council missing": 100 * sc_na.mean(),
    "suburb_group = OTHER": 100 * (sg == "OTHER").mean(),
    "suburb_group = '#'": 100 * (sg == "#").mean(),
    "Complaint type missing": 100 * train["C3_Complaint_Type"].isna().mean(),
})
fig, ax = plt.subplots(figsize=(8.6, 4.8))
miss.iloc[::-1].plot(kind="barh", ax=ax, color="#4c78a8")
ax.set_title("Figure 2.2.2 Missingness and grouped-location shares (train, percent of rows)")
ax.set_xlabel("Percent of training rows")
ax.set_xlim(0, 50)
for i, v in enumerate(miss.iloc[::-1].values):
    ax.text(v + 0.6, i, f"{v:.2f}%", va="center", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
save(fig, "fig02_missingness_quality.png")


**Interpretation (2.2.2).** Ward 40.56% and Sub_Council 41.55% missing on train. Do not mode-impute a ward. `MISSING` is a category. `#` in suburb_group maps to `MISSING` in `time_features`.


In [ ]:
# 2.2.3 feature–target 1: department
dept_n = train["department"].astype(str).value_counts()
order = dept_n.index.tolist()
ct = pd.crosstab(train["department"].astype(str), train["target"].astype(str), normalize="index")
ct = ct.reindex(index=order, columns=CLASSES)
labels = [f"{d}  (n={dept_n[d]:,})" for d in order]
fig, ax = plt.subplots(figsize=(10.2, 6.2))
left = np.zeros(len(order))
y = np.arange(len(order))
for cls, col in zip(CLASSES, COLORS):
    vals = ct[cls].values
    ax.barh(y, vals, left=left, color=col, label=cls, height=0.72)
    left = left + vals
ax.set_yticks(y, labels)
ax.set_xlabel("Share of training requests (proportion)")
ax.set_title("Figure 2.2.3 Resolution class mix by department (train)")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
save(fig, "fig03_department_class_mix.png")


**Interpretation (2.2.3).** Department is known at logging and changes the mix. Electricity is mostly Fast; Public housing and Open space are majority Unresolved on train (closing practice as well as delay). This is EDA, not a second model.


In [ ]:
# 2.2.4 feature–target 2: complaint type (top 12 by train volume)
top = train["C3_Complaint_Type"].astype(str).value_counts().head(12)
sub = train[train["C3_Complaint_Type"].astype(str).isin(top.index)]
order_t = top.index.tolist()
ct2 = pd.crosstab(sub["C3_Complaint_Type"].astype(str), sub["target"].astype(str), normalize="index")
ct2 = ct2.reindex(index=order_t, columns=CLASSES)
labels_t = [f"{t}  (n={int(top[t]):,})" for t in order_t]
fig, ax = plt.subplots(figsize=(11.0, 6.8))
left = np.zeros(len(order_t))
y = np.arange(len(order_t))
for cls, col in zip(CLASSES, COLORS):
    vals = ct2[cls].values
    ax.barh(y, vals, left=left, color=col, label=cls, height=0.72)
    left = left + vals
ax.set_yticks(y, labels_t)
ax.set_xlabel("Share of training requests (proportion)")
ax.set_title("Figure 2.2.4 Resolution class mix for the 12 most common complaint types (train)")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
save(fig, "fig04_complaint_type_class_mix.png")


**Interpretation (2.2.4).** No Power is ~92% Fast; underground water leaks and burst pipes are much slower. Complaint type at logging is a real signal. Rare types (<100 train rows) are grouped infrequent in the encoder.


In [ ]:
# 2.2.5 temporal / subgroup view (required)
ts = pd.to_datetime(train["Created_On_Date"])
monthly = (
    train.assign(_month=ts.dt.to_period("M").astype(str))
    .groupby(["_month", "target"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASSES)
)
fig, ax = plt.subplots(figsize=(11.4, 5.4))
monthly.plot(kind="area", stacked=True, ax=ax, color=COLORS, alpha=0.88, linewidth=0)
ax.set_title("Figure 2.2.5 Train tickets by created month and class (stratified random split, 2023–2026)")
ax.set_xlabel("Month created (Africa/Johannesburg date)")
ax.set_ylabel("Number of service requests")
ticks = list(range(0, len(monthly), 3))
ax.set_xticks(ticks)
ax.set_xticklabels([monthly.index[i] for i in ticks], rotation=45, ha="right")
ax.legend(title="Resolution class")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
save(fig, "fig05_temporal_train.png")
print("train created span", ts.min().date(), "→", ts.max().date())


**Interpretation (2.2.5).** Main split is stratified **random**, so train still spans 1 Jan 2023 – 20 May 2026. The Unresolved band thickens later — that is why a time split is robustness only (2.3), not the champion protocol. Test is not in this chart.


## 2.3 Split data before learning transformations. Justify train/validation/test or nested cross-validation design, including stratification/group/time constraints. Reserve the test set for final evaluation only. (3)


In [ ]:
test = pd.read_parquet(DATA / "test.parquet")  # split table only; no fit, no plots

def split_row(df, name):
    n = len(df)
    vc = df["target"].astype(str).value_counts().reindex(CLASSES)
    t = pd.to_datetime(df["Created_On_Date"])
    return {
        "set": name,
        "n": n,
        "Fast_%": 100 * vc["Resolved Fast"] / n,
        "Slow_%": 100 * vc["Resolved Slow"] / n,
        "Unresolved_%": 100 * vc["Unresolved"] / n,
        "created_min": t.min().date().isoformat(),
        "created_max": t.max().date().isoformat(),
    }

main = pd.DataFrame([
    split_row(train, "train"),
    split_row(val, "val"),
    split_row(test, "test (locked)"),
])
print("MAIN SPLIT — stratified random 70/15/15 on target, seed", prep.SEED)
display(main.round(2))

time_parts = []
for fname, label in (
    ("time_train.parquet", "time_train"),
    ("time_val.parquet", "time_val"),
    ("time_test.parquet", "time_test"),
):
    p = DATA / fname
    if p.exists():
        time_parts.append(split_row(pd.read_parquet(p), label))
if time_parts:
    print("TIME SPLIT — robustness only (not tuning)")
    display(pd.DataFrame(time_parts).round(2))

tr_types = set(train["C3_Complaint_Type"].astype(str))
print("unseen complaint type share val %:",
      round(100 * (~val["C3_Complaint_Type"].astype(str).isin(tr_types)).mean(), 4))
print("Nested CV: not used. One frozen three-way split, seed 42.")
print("Test reserved: preprocessor is fitted on TRAIN only in 2.4. Test is not passed to fit.")


**Justification.** Stratification keeps 67.4 / 21.5 / 11.1 in every main fold. Time split Unresolved jumps 7.4% → ~19.8% (closing drift), so **main = random**, **time = robustness**. No grouping split on suburb: those fields are features. Test locked for final evaluation (Q5), not for EDA or encoder fit.


## 2.4 Implement cleaning, encoding, scaling, imputation, feature engineering and imbalance treatment inside a reproducible pipeline. Explain choices and prevent leakage. (5)

`prep.build_preprocessor()`: `FunctionTransformer(time_features)` (clean + calendar engineer) → one-hot (`min_frequency=100`, sparse) → `StandardScaler` on `is_weekend`. Missing location → `MISSING` (no mode impute). Imbalance: class weights on later estimators, not a rewrite of these folds.

This cell **fits the preprocessor on train only** and transforms val. It does **not** fit LR / Forest / MLP.


In [ ]:
from sklearn.pipeline import Pipeline

assert not prep.BANNED & set(prep.FEATURES)

# Cleaning + engineering sample (not fitted — pure function)
engineered = prep.time_features(train[prep.FEATURES].head(3))
print("after time_features (clean + month/weekday/is_weekend):")
display(engineered)

# Fit on TRAIN only
preprocessor = prep.build_preprocessor()
Xt = preprocessor.fit_transform(train[prep.FEATURES])
Xv = preprocessor.transform(val[prep.FEATURES])
print("train matrix", Xt.shape, "val matrix", Xv.shape, "sparse", hasattr(Xt, "toarray"))

# Unknown handling: val-only transform, no refit
print("encoder fitted on train; val transformed with the frozen encoder.")

# Imbalance plan (no resampling of val/test)
print("imbalance: class_weight='balanced' on later models; resampling only as a train-fold ablation.")
print("shared pipeline object:", Pipeline([("prep", preprocessor)]))
print("leakage check — columns used:", prep.FEATURES)
print("leakage check — banned:", sorted(prep.BANNED))


Choices: one sklearn `Pipeline` so Dummy and later models see the same columns; sparse one-hot (~1,083 cols) because 3.0M dense dummies will not fit; `min_frequency=100` from **train** counts; weekend scaled; categoricals not. Test is still locked — not fitted, not used to choose this encoder.

Q3 (four substantive models) is **not** in this notebook.
